# 🎬 Promo Extractor (Kuku · Problem 1)
Finds 20–60 s promo clips in a microdrama screenplay: a hook in the first 3 s, a cliffhanger at the cut, and clear to someone who has never seen the show.

**Two modes, same pipeline.** *Rules mode* (default, no API key) finds hooks and cliffhangers from the script's own signals: bracketed emotions, action verbs, sound cues, `!`, `?` and trailing `...`. *LLM mode* (section 8, optional) swaps in a model (OpenAI, Claude or Gemini) for the hook-finding, judging and ranking steps. In both, Python owns every timestamp and enforces the 20–60 s and 3-second rules.

Run the cells in order. No API key is needed for sections 1–7.

## 1. Get the code

In [ ]:
REPO = "https://github.com/ShubNoob/KUKU_PROM_NOAPI"

import glob, os, shutil
%cd /content
for z in glob.glob("/content/repo/**/*.zip", recursive=True):   # keep an uploaded scripts zip
    shutil.copy(z, "/content/")
!rm -rf repo && git clone -q $REPO repo
hits = glob.glob("/content/repo/**/run.py", recursive=True)
assert hits, "run.py not found - check the files are on GitHub"
%cd {os.path.dirname(hits[0])}
!pip -q install pydantic pyyaml
!ls

## 2. Episode scripts (.zip of .txt files)
Uses a scripts zip already in `/content`, otherwise asks you to upload it. Code zips are ignored.

In [ ]:
import glob, os, shutil, zipfile
from google.colab import files

def episode_count(path):
    """How many episode .txt scripts a zip holds (0 if it is not a scripts zip)."""
    try:
        names = zipfile.ZipFile(path).namelist()
    except (zipfile.BadZipFile, FileNotFoundError):
        return 0
    return sum(n.lower().endswith(".txt") and "episode" in os.path.basename(n).lower()
               and "__macosx" not in n.lower() for n in names)

zips = [z for z in glob.glob("/content/*.zip") if episode_count(z) >= 5]
if zips:
    ZIP = zips[0]
else:
    print("No scripts zip found. Upload the zip of episode .txt files (NOT the code zip):")
    name = next(iter(files.upload()))
    ZIP = shutil.copy(os.path.abspath(name), "/content/")   # keep it safe in /content
    assert episode_count(ZIP) >= 5, f"{name} has no episode scripts - wrong zip?"
print(f"Scripts: {ZIP} ({episode_count(ZIP)} episode files)")

## 3. Checks (16 automated tests)
Timestamp checks, parsing, and model handling (retries, fallbacks, OpenAI / Claude response formats with stand-in servers).

In [ ]:
!python test_checks.py | tail -1

## 4. Run on every episode (rules mode, no API key)

In [ ]:
!python run.py "$ZIP" --provider rules --out outputs

## 5. Results

In [ ]:
from IPython.display import Markdown, display
import json
log = json.load(open("outputs/run_log.json"))
print("Episodes:", log["episodes"], "| hooks:", log["hooks_found"], "| clips judged:", log["clips_judged"],
      "| tiers:", log["tier_counts"], "| rejected by checks:", log["rejected_before_judging"])
display(Markdown(open("outputs/shortlist.md").read()))

## 6. Download the raw outputs and the hand-check sheet

In [ ]:
from google.colab import files
!zip -qr raw_outputs.zip outputs
files.download("raw_outputs.zip")
files.download("outputs/hand_check.csv")

## 7. Hand check (human in the loop)
Open `hand_check.csv`, read each clip in its episode's script between `start_ts` and `end_ts`, fill `human_good` (y/n) and `human_reason`, save as CSV, then upload it here. The sheet hides the scores so you judge first.

In [ ]:
labelled = next(iter(files.upload()))
!python run.py --score "$labelled"

## 8. Optional: LLM mode (needs an API key)
Add `OPENAI_API_KEY` in the 🔑 Secrets panel (with *Notebook access* on). For Claude use `--provider anthropic` and `ANTHROPIC_API_KEY`; for Gemini `--provider gemini` and `GEMINI_API_KEY`. Results go to `outputs_llm/`, so they can be compared with the rules run.

In [ ]:
import os
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
!pip -q install openai
MODEL, FALLBACK = "gpt-5-mini", "gpt-4o-mini"      # use model names your key can access
!python run.py "$ZIP" --provider openai --model $MODEL --fallback "$FALLBACK" --episodes 1-3 --out outputs_llm